# Step 1: Fine-tune Qwen3-8B Base with Unsloth + QLoRA

**Before running anything:**
- Go to Runtime > Change runtime type > Select T4 GPU (free) or A100 (Colab Pro)
- Upload your train.jsonl file using the file upload cell below

**What this notebook does:**
1. Installs Unsloth and dependencies
2. Loads Qwen3-8B Base with 4-bit quantization
3. Attaches LoRA adapters
4. Trains on your train.jsonl
5. Saves the adapter weights to Google Drive

In [ ]:
# CELL 1: Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# CELL 2: Install Unsloth and required packages
# This takes 3-5 minutes. Do not interrupt.
!pip install unsloth --quiet
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps trl peft accelerate bitsandbytes --quiet
!pip install datasets --quiet
print("Installation complete.")

In [ ]:
# CELL 3: Upload your train.jsonl file
# Run this cell, then click the upload button that appears and select train.jsonl
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

In [ ]:
# CELL 4: Verify the dataset
import json

records = []
with open('train.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Total training examples: {len(records)}")
print("\nFirst example:")
print("USER:", records[0]['messages'][0]['content'][:200])
print("ASSISTANT:", records[0]['messages'][1]['content'][:200])

In [ ]:
# CELL 5: Load Qwen3-8B Base with Unsloth (4-bit quantized)
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None          # None = auto-detect (float16 on T4, bfloat16 on A100)
LOAD_IN_4BIT = True   # QLoRA: quantize base model to 4-bit to save VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-Base",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

print("Model loaded successfully.")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# CELL 6: Attach LoRA adapters
# Only these adapter weights are trained, not the full 8B model.
# This reduces trainable parameters from ~8B to ~40M.

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                          # LoRA rank. Higher = more capacity, more VRAM.
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,                 # Scaling factor. Keep equal to r.
    lora_dropout = 0,                # 0 is optimized in Unsloth
    bias = "none",                   # No bias terms in adapters
    use_gradient_checkpointing = "unsloth",  # Saves VRAM
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable / 1e6:.2f}M / {total / 1e9:.2f}B total")
print(f"Trainable %: {100 * trainable / total:.4f}%")

In [ ]:
# CELL 7: Format dataset using the chat template
from datasets import Dataset

def format_example(example):
    """
    Convert messages list into a single training string
    using the model's built-in chat template.
    """
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize = False,
        add_generation_prompt = False
    )
    return {"text": text}

# Load the JSONL into a HuggingFace Dataset
raw_data = []
with open('train.jsonl', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_example, remove_columns=['messages'])

print(f"Dataset size: {len(dataset)}")
print("\nFormatted example (first 500 chars):")
print(dataset[0]['text'][:500])

In [ ]:
# CELL 8: Set up the trainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,  # Set True if examples are short to pack multiple per batch
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Effective batch size = 2 * 4 = 8
        warmup_steps = 10,
        num_train_epochs = 3,              # 3 full passes over the data
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",             # Memory-efficient optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "./outputs",
        report_to = "none",               # Disable wandb/tensorboard
    ),
)

print("Trainer configured.")

In [ ]:
# CELL 9: Check VRAM before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Total VRAM: {max_memory} GB")
print(f"Currently reserved: {start_gpu_memory} GB")
print(f"Available for training: {max_memory - start_gpu_memory} GB")

In [ ]:
# CELL 10: TRAIN
# On a free T4 GPU with 145 examples and 3 epochs, this takes roughly 15-25 minutes.
# Watch the loss column — it should decrease over steps.

print("Starting training...")
trainer_stats = trainer.train()

# Print final stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\nTraining complete.")
print(f"Total training time: {trainer_stats.metrics['train_runtime']:.0f} seconds")
print(f"Final training loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"Peak VRAM used: {used_memory} GB")

In [ ]:
# CELL 11: Save adapter weights locally first
model.save_pretrained("qwen3_ioai_lora")
tokenizer.save_pretrained("qwen3_ioai_lora")
print("Saved locally to ./qwen3_ioai_lora")

import os
files_saved = os.listdir('qwen3_ioai_lora')
print("Files saved:", files_saved)

In [ ]:
# CELL 12: Mount Google Drive and save weights there
# This is important: Colab resets after ~12 hours and you will lose local files.
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_path = '/content/drive/MyDrive/qwen3_ioai_lora'

if os.path.exists(save_path):
    shutil.rmtree(save_path)

shutil.copytree('qwen3_ioai_lora', save_path)
print(f"Weights saved to Google Drive at: {save_path}")
print("Files:", os.listdir(save_path))

In [ ]:
# CELL 13: Quick sanity check — run one inference to make sure the model works
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)  # Enable faster inference mode

test_question = """A model achieves 99% accuracy on training data but only 62% on test data. Which of the following best describes this situation?
A) Underfitting due to high bias
B) Overfitting due to high variance
C) Good generalization
D) Data leakage"""

messages = [{"role": "user", "content": test_question}]

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

output = model.generate(
    input_ids,
    max_new_tokens = 300,
    temperature = 0.1,
    do_sample = True,
)

response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
print("Question:", test_question)
print("\nModel response:")
print(response)